In [1]:
# Position class

from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from math import sqrt, isclose, cos, sin, isnan, isinf

# from src.main.world_objects.robot_objects.degrees import Degrees


class InvalidPositionError(Exception):
    """Custom exception for invalid position coordinates."""
    pass


@dataclass(frozen=True)
class Position:
    x: float
    y: float

    def __post_init__(self):
        # Check if coordinates are numeric
        if not isinstance(self.x, (int, float)) or not isinstance(self.y, (int, float)) or isnan(self.x) or isnan(self.y) or isinf(self.x) or isinf(self.y):  #
            raise InvalidPositionError(
                f"Coordinates must be numeric, got x: {type(self.x).__name__}, y: {type(self.y).__name__}."
            )

    @lru_cache(maxsize=None)
    def distance_to(self, other: "Position") -> float:
        """Calculate the distance to another Position."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return sqrt((self.x - other.x) ** 2 + (self.y - other.y) ** 2)

    def move(self, angle: "Degrees"| int, steps: float) -> "Position":
        """Move the position by a certain number of steps in the specified angle."""
        if not isinstance(angle, Degrees):  #
            raise ValueError("Angle must be a Degrees instance.")

        rad_angle = angle.to_radians()
        return Position(self.x + steps * cos(rad_angle), self.y + steps * sin(rad_angle))

    def is_in(self, top_left: "Position", bottom_right: "Position") -> bool:
        """Check if the position is within a defined rectangular area."""
        if not all(isinstance(p, Position) for p in [top_left, bottom_right]):  #
            raise ValueError("Top left and bottom right must be Position instances.")
        return (
            top_left.x <= self.x <= bottom_right.x
            and bottom_right.y <= self.y <= top_left.y
        )

    def __add__(self, other: "Position") -> "Position":
        """Add two positions."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return Position(self.x + other.x, self.y + other.y)

    def __sub__(self, other: "Position") -> "Position":
        """Subtract two positions."""
        if not isinstance(other, Position):  #
            raise ValueError("The argument must be a Position instance.")
        return Position(self.x - other.x, self.y - other.y)

    def __eq__(self, other: object) -> bool:
        """Check equality of two positions."""
        if not isinstance(other, Position):
            return NotImplemented
        return isclose(self.x, other.x, abs_tol=1e-9) and isclose(
            self.y, other.y, abs_tol=1e-9
        )

    def surrounding(self) -> ["Position"]:
        return [
            (self.x - 1, self.y + 1), (self.x + 1, self.y - 1),  # alt diag
            (self.x, self.y + 1), (self.x, self.y - 1),  # vertical
            (self.x + 1, self.y), (self.x - 1, self.y), # horizontal
            (self.x + 1, self.y + 1), (self.x - 1, self.y - 1),  # main diag
        ]

    def as_tuple(self):
        return self.x, self.y

    def __repr__(self):
        return f"Position(x={self.x}, y={self.y})"

    def __str__(self) -> str:
        return f"({self._format_value(self.x)}, {self._format_value(self.y)})"

    @staticmethod
    def _format_value(value: float) -> str:
        """Format value for string representation."""
        if abs(value - round(value)) < 1e-6:
            return f"{round(value)}"
        else:
            return f"{value:.6f}"
            

In [2]:
# Degrees class

from dataclasses import dataclass, field
from math import radians, isclose, isinf, isnan
from functools import lru_cache

class InvalidAngleError(Exception):
    """Custom exception for invalid angle values."""
    pass

@dataclass(frozen=True)
class Degrees:
    _angle: float = field(repr=False)

    def __post_init__(self):
        # Normalize the angle using the setter


        if not isinstance(self._angle, (int, float)) or isinf(self.angle) or isnan(self.angle):
            raise InvalidAngleError

        object.__setattr__(self, '_angle', self._normalize_angle(self._angle))

    @property
    def angle(self) -> float:
        """Get the normalized angle."""
        return self._angle

    @staticmethod
    @lru_cache(maxsize=None)
    def _normalize_angle(angle: float) -> float:
        """Normalize the angle to be within [0, 360) degrees."""
        return angle % 360

    def turn_left(self, degrees: float = 90) -> 'Degrees':
        """Turn left by the given number of degrees."""
        return Degrees(self._normalize_angle(self.angle - degrees))

    def turn_right(self, degrees: float = 90) -> 'Degrees':
        """Turn right by the given number of degrees."""
        return Degrees(self._normalize_angle(self.angle + degrees))

    def to_radians(self) -> float:
        """Convert degrees to radians."""
        return radians(self.angle)

    def __add__(self, other: 'Degrees') -> 'Degrees':
        """Add two Degrees instances."""
        if not isinstance(other, Degrees): #
            raise ValueError(f"Invalid addition: {self.angle} + {other.angle} results in an invalid angle.")
        return Degrees(self._normalize_angle(self.angle + other.angle))

    def __sub__(self, other: 'Degrees') -> 'Degrees':
        """Subtract two Degrees instances."""
        if not isinstance(other, Degrees): #
            raise ValueError(f"Invalid subtraction: {self.angle} - {other.angle} results in an invalid angle.")

        return Degrees(self._normalize_angle(self.angle - other.angle))

    def __eq__(self, other: object) -> bool:
        """Check equality of two Degrees instances."""
        if not isinstance(other, Degrees):
            return NotImplemented
        return isclose(self.angle, other.angle, abs_tol=1e-9)

    def __hash__(self) -> int:
        """Return a hash of the angle for hashable collections."""
        return hash(round(self.angle, 9))

    def __repr__(self) -> str:
        return f"Degrees(angle={self.angle})"

    def __str__(self) -> str:
        return f"{self.angle}°"

    def to_cardinal(self) -> str:
        """Convert angle to cardinal direction."""
        cardinal_points = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
        index = round(self.angle / 45) % 8
        return cardinal_points[index]
        